# Fairness Metrics for Predictive Models

## Requirements

In [18]:
from aif360.datasets import GermanDataset
from aif360.sklearn.metrics import statistical_parity_difference, disparate_impact_ratio, equal_opportunity_difference, average_odds_error

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import pandas as pd 
import numpy as np

from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

In [19]:
np.__version__

'2.1.1'

## Initial elements

### Data


In [20]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [21]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0



### Response and predictors


In [22]:
quant_predictors = ['month', 'credit_amount', 'investment_as_income_percentage',
                    'residence_since', 'number_of_credits', 'people_liable_for']
cat_predictors = [x for x in german_data_pd.columns if x not in quant_predictors + [response_name]]
predictors = quant_predictors + cat_predictors

In [23]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute


### Outer evaluation method: simple-validation (train-test split)

In [24]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)

### Fairness metrics


In [25]:
logistic_reg = LogisticRegression(solver='liblinear', random_state=123)
logistic_reg.fit(X_train, Y_train)
Y_test_hat = logistic_reg.predict(X_test)
A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set


#### Statistical parity difference (Independence)


In [26]:
value = statistical_parity_difference(y_true=Y_test,
                                      y_pred=Y_test_hat, 
                                      prot_attr=A_test,
                                      priv_group=sens_privileged_groups[0][sens_variable_name], 
                                      pos_label=response_favorable_label)
value

np.float64(-0.32539682539682535)

`statistical_parity_difference`  $\hspace{0.05cm}=\hspace{0.05cm} P\left(\widehat{Y} = + \hspace{0.1cm}|\hspace{0.1cm} A = \text{unprivileged}\right) - P\left(\widehat{Y} = + \hspace{0.1cm}|\hspace{0.1cm} A = \text{privileged}\right)$

**Properties**

- $\in [-1, 1]$

- The closer to $-1$, the more favorable are the results (predictions) of the models to the privileged class of the sensitive variable.

- The closer to $1$, the more favorable are the results (predictions) of the models to the unprivileged class of the sensitive variable.

- The closer to $0$, the more fairness (parity) in the model results (predictions).


**Doubts**

- No veo que esta metrica sea una medida de justicia en todos los casos. Ejemplo: concesion de creditos, variable sensible color de piel (blanco, negro). Imaginemos que el grupo de los negros reune una serie de condiciones que les hace objetivamente mas riesgosos que los blancos (peor situacion economica, mayores deudas etc). Que en este caso el modelo conceda significativamente mas creditos a los blancos que a los negros no seria injusto, no? Puede setar basado en criterios economicos objetivos que estan asociados a esos grupos, y no por el color de piel en si, no??


---


#### Absolute statistical parity difference (Absolute Independence)


In [27]:
def abs_statistical_parity_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    value = statistical_parity_difference(y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label)
    return np.abs(value)

In [28]:
value = abs_statistical_parity_difference(y_true=Y_test,
                                          y_pred=Y_test_hat, 
                                          prot_attr=A_test,
                                          priv_group=sens_privileged_groups[0][sens_variable_name], 
                                          pos_label=response_favorable_label)
value

np.float64(0.32539682539682535)

`abs_statistical_parity_difference`  $\hspace{0.05cm}=\hspace{0.05cm} \left| P\left(\widehat{Y} = + \hspace{0.1cm}|\hspace{0.1cm} A = \text{unprivileged}\right) - P\left(\widehat{Y} = + \hspace{0.1cm}|\hspace{0.1cm} A = \text{privileged}\right) \right|$

- $P(\hat{Y} = + \hspace{0.08cm} | \hspace{0.08cm} A = \text{unprivileged})= \text{Proportion of credit granted to black people}$

- $P(\hat{Y} = + \hspace{0.08cm} | \hspace{0.08cm} A = \text{privileged})= \text{Proportion of credit granted to white people}$

**Properties**

- $\in [0,1]$
- The **closer to $0$** the less parity difference between the sensitive groups (**more fairness** (?)).
- The **closer to $1$** the more parity difference between the sensitive groups (**less fairness** (?)).


---


#### Disparate impact ratio


In [29]:
value = disparate_impact_ratio(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test,
                               priv_group=sens_privileged_groups[0][sens_variable_name], 
                               pos_label=response_favorable_label)
value

0.6057692307692308

  `disparate_impact_ratio` $\hspace{0.05cm}=\hspace{0.05cm}\dfrac{P(\hat{Y} =+ \hspace{0.08cm} |\hspace{0.08cm} A = \text{unprivileged})}
{Pr(\hat{Y} = + \hspace{0.08cm}|\hspace{0.08cm} A = \text{privileged})}$

- $P(\hat{Y} = + \hspace{0.08cm}|\hspace{0.08cm} A = \text{unprivileged})= \text{Proportion of credit granted to black people}$

- $P(\hat{Y} = + \hspace{0.08cm}|\hspace{0.08cm} A = \text{privileged})= \text{Proportion of credit granted to white people}$

**Properties**

- $\in [0,\infty]$
- The **closer to $1$**, the less parity difference between the sensitive groups (**more fairness**).
- The **closer to $0$ or to $\infty$** (**farther from $1$**) , the more parity difference between the sensitive groups (**less fairness**).

---

#### Absolute equal opportunity difference

In [30]:
def abs_equal_opportunity_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    value = equal_opportunity_difference(y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label)
    return np.abs(value)

In [31]:
abs_equal_opportunity_difference(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test, 
                             priv_group=sens_privileged_groups[0][sens_variable_name], 
                             pos_label=response_favorable_label)


np.float64(0.320855614973262)

`equal_opportunity_difference` $\hspace{0.1cm}=\hspace{0.1cm} \left| \hspace{0.05cm} P( \hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=+\hspace{0.04cm},\hspace{0.08cm} A = \text{unprivilaged}) \hspace{0.1cm}-\hspace{0.1cm} P( \hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm}  Y=+\hspace{0.04cm},\hspace{0.08cm}  A = \text{privilaged}) \hspace{0.05cm}\right| \\[0.5cm] \hspace{4.85cm} = \Big| \hspace{0.05cm} TPR_{A=\text{unpriv}}\hspace{0.1cm}-\hspace{0.1cm} TPR_{A=\text{priv}} \hspace{0.05cm} \Big |  \\[0.5cm] \hspace{4.85cm} = \Big| \hspace{0.05cm} FNR_{A=\text{unpriv}}\hspace{0.1cm}-\hspace{0.1cm} FNR_{A=\text{priv}} \hspace{0.05cm} \Big|$

Taking into account that TPR = 1 - FNR

- $P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=+\hspace{0.04cm},\hspace{0.08cm} A = \text{unprivilaged}) = \text{Proportion of credits granted by the model (predicted positive) among black people (unprivileged) that actually deserved it (true positive)} = TPR_{A=\text{unprivilaged}}$

- $P( \hat{Y}=+\hspace{0.1cm}|\hspace{0.1cm} Y=+\hspace{0.04cm},\hspace{0.08cm} A = \text{unprivilaged}) = \text{Proportion of credit granted by the model  (predicted positive) among white people (privileged) that actually deserved it (true positive)} = TPR_{A=\text{privilaged}}$

**Properties**

- $\in [0, 1]$
- The **closer to $0$**, the less difference of opportunity (e.g. for getting the credit) between the sensitive groups (**more fairness**).
- The **closer to $1$**, the more difference of opportunity (e.g. for getting the credit) between the sensitive groups (**less fairness**).


---

#### Average odds error (Separation)

In [32]:
average_odds_error(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test, 
                     priv_group=sens_privileged_groups[0][sens_variable_name], 
                     pos_label=response_favorable_label)

np.float64(0.3333601383136987)

`average_fpr_tpr_error` $\hspace{0.1cm}=\hspace{0.1cm} \dfrac{|P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=- \hspace{0.08cm},\hspace{0.08cm} A = \text{unpriv})  - P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=- \hspace{0.08cm},\hspace{0.08cm} A = \text{priv})| + |P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=+\hspace{0.08cm},\hspace{0.08cm} A = \text{unpriv})  - P(\hat{Y}=+ \hspace{0.1cm}|\hspace{0.1cm} Y=+ \hspace{0.08cm},\hspace{0.08cm} A  = \text{priv}) |}{2} \\[0.3cm] \hspace{3.7cm} = \hspace{0.1cm} \dfrac{|FPR_{A = \text{unpriv}} - FPR_{A = \text{priv}}| + |TPR_{A = \text{unpriv}} - TPR_{A = \text{priv}}|}{2} \\[0.3cm] \hspace{3.7cm} = \hspace{0.1cm} \dfrac{|FPR_{A = \text{unpriv}} - FPR_{A = \text{priv}}| + |FNR_{A = \text{priv}} - FNR_{A = \text{unpriv}}|}{2} $


Taking into account that TPR = 1 - FNR

**Properties**

- $\in [0, 1]$
- The **closer to $0$**, the less differences in how the model treats the sensitive groups (**more fairness**).
- The **closer to $1$**, the more differences in how the model treats the sensitive groups (**less fairness**).


---

#### False/True positive/negative rate difference

`false_positive_rate_difference` $= FPR_{A = \text{unpriv}} - FPR_{A = \text{priv}}$

`false_negative_rate_difference` $= FNR_{A = \text{unpriv}} - FPR_{A = \text{priv}}$

`true_positive_rate_difference` $= TPR_{A = \text{unpriv}} - TPR_{A = \text{priv}} = - $`false_negative_rate_difference`

`true_negative_rate_difference` $= TNR_{A = \text{unpriv}} - TNR_{A = \text{priv}} = - $`false_positive_rate_difference`

In [ ]:
def check_data_type(y_true, prot_attr):

    if isinstance(y_true, pd.Series):
        y_true = y_true.to_numpy()
    if isinstance(prot_attr, pd.Series):
        prot_attr = prot_attr.to_numpy()  

    return y_true, prot_attr

In [ ]:
def get_neg_label(y_true, pos_label):

    unique_y_true = np.unique(y_true)
    if len(unique_y_true) == 1:
        raise ValueError("y_true contains only one unique value, unable to determine negative label.")
    else:
        neg_label_list = [x for x in unique_y_true if x != pos_label]
        neg_label = neg_label_list[0] if len(unique_y_true) == 2 else neg_label_list
    
    return neg_label

In [ ]:
def get_unpriv_group(prot_attr, priv_group):

    unique_prot_attr = np.unique(prot_attr)
    if len(unique_prot_attr) == 1:
        raise ValueError("prot_attr contains only one unique value, unable to determine unprivileged group.")  
    else:
        unpriv_group_list = [x for x in unique_prot_attr if x != priv_group]
        unpriv_group = unpriv_group_list[0] if len(unique_prot_attr) == 2 else unpriv_group_list

    return unpriv_group

In [ ]:
def false_positive_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label):

    y_true, prot_attr = check_data_type(y_true, prot_attr)
    neg_label = get_neg_label(y_true, pos_label)

    true_negative_idx = np.where(y_true == neg_label)[0]  if len(np.unique(y_true)) == 2 else np.where(y_true in neg_label)[0] 
    predicted_positive_idx = np.where(y_pred == pos_label)[0]  
    priv_idx = np.where(prot_attr == priv_group)[0] 
    true_negative_priv_idx = np.intersect1d(true_negative_idx, priv_idx)
    true_negative_priv_pred_pos_idx = np.intersect1d(true_negative_priv_idx, predicted_positive_idx)
    FPR_priv = len(true_negative_priv_pred_pos_idx) / len(true_negative_priv_idx)

    return FPR_priv 

In [ ]:
def false_positive_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label):

    y_true, prot_attr = check_data_type(y_true, prot_attr)
    neg_label = get_neg_label(y_true, pos_label)
    unpriv_group = get_unpriv_group(prot_attr, priv_group)

    true_negative_idx = np.where(y_true == neg_label)[0]  if len(np.unique(y_true)) == 2 else np.where(y_true in neg_label)[0] 
    predicted_positive_idx = np.where(y_pred == pos_label)[0]  
    unpriv_idx = np.where(prot_attr == unpriv_group)[0] if len(np.unique(prot_attr)) == 2 else np.where(prot_attr in unpriv_group)[0]
    true_negative_unpriv_idx = np.intersect1d(true_negative_idx, unpriv_idx)
    true_negative_unpriv_pred_pos_idx = np.intersect1d(true_negative_unpriv_idx, predicted_positive_idx)
    FPR_unpriv = len(true_negative_unpriv_pred_pos_idx) / len(true_negative_unpriv_idx)

    return FPR_unpriv 

In [ ]:
def false_negative_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label):

    y_true, prot_attr = check_data_type(y_true, prot_attr)
    neg_label = get_neg_label(y_true, pos_label)
       
    true_positive_idx = np.where(y_true == pos_label)[0]    
    predicted_negative_idx = np.where(y_pred == neg_label)[0] if len(np.unique(y_true)) == 2 else  np.where(y_true in neg_label)[0]     
    priv_idx = np.where(prot_attr == priv_group)[0] if len(np.unique(prot_attr)) == 2 else np.where(prot_attr in priv_group)[0]
    true_positive_priv_idx = np.intersect1d(true_positive_idx, priv_idx)
    true_positive_priv_pred_neg_idx = np.intersect1d(true_positive_priv_idx, predicted_negative_idx)
    FNR_priv = len(true_positive_priv_pred_neg_idx) / len(true_positive_priv_idx)

    return FNR_priv

In [ ]:
def false_negative_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label):

    y_true, prot_attr = check_data_type(y_true, prot_attr)
    neg_label = get_neg_label(y_true, pos_label)
    unpriv_group = get_unpriv_group(prot_attr, priv_group)
       
    true_positive_idx = np.where(y_true == pos_label)[0]    
    predicted_negative_idx = np.where(y_pred == neg_label)[0] if len(np.unique(y_true)) == 2 else  np.where(y_true in neg_label)[0]     
    unpriv_idx = np.where(prot_attr == unpriv_group)[0] if len(np.unique(prot_attr)) == 2 else np.where(prot_attr in unpriv_group)[0]
    true_positive_unpriv_idx = np.intersect1d(true_positive_idx, unpriv_idx)
    true_positive_unpriv_pred_neg_idx = np.intersect1d(true_positive_unpriv_idx, predicted_negative_idx)
    FNR_unpriv = len(true_positive_unpriv_pred_neg_idx) / len(true_positive_unpriv_idx)

    return FNR_unpriv

In [ ]:
def true_negative_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label):
    
    y_true, prot_attr = check_data_type(y_true, prot_attr)
    neg_label = get_neg_label(y_true, pos_label)
    
    if len(np.unique(y_true)) == 2:
        true_negative_idx = np.where(y_true == neg_label)[0]   
        predicted_negative_idx = np.where(y_pred == neg_label)[0]  
    else:
        true_negative_idx = np.where(y_true in neg_label)[0] 
        predicted_negative_idx = np.where(y_pred in neg_label)[0] 
    priv_idx = np.where(prot_attr == priv_group)[0] 
    true_negative_priv_idx = np.intersect1d(true_negative_idx, priv_idx)
    true_negative_priv_pred_neg_idx = np.intersect1d(true_negative_priv_idx, predicted_negative_idx)
    TNR_priv = len(true_negative_priv_pred_neg_idx) / len(true_negative_priv_idx)

    return TNR_priv

In [ ]:
def true_negative_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label):
    
    y_true, prot_attr = check_data_type(y_true, prot_attr)
    neg_label = get_neg_label(y_true, pos_label)
    unpriv_group = get_unpriv_group(prot_attr, priv_group)

    if len(np.unique(y_true)) == 2:
        true_negative_idx = np.where(y_true == neg_label)[0]   
        predicted_negative_idx = np.where(y_pred == neg_label)[0]  
    else:
        true_negative_idx = np.where(y_true in neg_label)[0] 
        predicted_negative_idx = np.where(y_pred in neg_label)[0] 
    unpriv_idx = np.where(prot_attr == unpriv_group)[0] if len(np.unique(prot_attr)) == 2 else np.where(prot_attr in unpriv_group)[0]
    true_negative_unpriv_idx = np.intersect1d(true_negative_idx, unpriv_idx)
    true_negative_unpriv_pred_neg_idx = np.intersect1d(true_negative_unpriv_idx, predicted_negative_idx)
    TNR_unpriv = len(true_negative_unpriv_pred_neg_idx) / len(true_negative_unpriv_idx)

    return TNR_unpriv

In [ ]:
def true_positive_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label):

    y_true, prot_attr = check_data_type(y_true, prot_attr)
    
    true_positive_idx = np.where(y_true == pos_label)[0]  
    predicted_positive_idx = np.where(y_pred == pos_label)[0]  
    priv_idx = np.where(prot_attr == priv_group)[0] 
    true_positive_priv_idx = np.intersect1d(true_positive_idx, priv_idx)
    true_positive_priv_pred_pos_idx = np.intersect1d(true_positive_priv_idx, predicted_positive_idx)
    TPR_priv = len(true_positive_priv_pred_pos_idx) / len(true_positive_priv_idx)    

    return TPR_priv

In [ ]:
def true_positive_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label):

    y_true, prot_attr = check_data_type(y_true, prot_attr)
    unpriv_group = get_unpriv_group(prot_attr, priv_group)
    
    true_positive_idx = np.where(y_true == pos_label)[0]  
    predicted_positive_idx = np.where(y_pred == pos_label)[0]  
    unpriv_idx = np.where(prot_attr == unpriv_group)[0] if len(np.unique(prot_attr)) == 2 else np.where(prot_attr in unpriv_group)[0]
    true_positive_unpriv_idx = np.intersect1d(true_positive_idx, unpriv_idx)
    true_positive_unpriv_pred_pos_idx = np.intersect1d(true_positive_unpriv_idx, predicted_positive_idx)
    TPR_unpriv = len(true_positive_unpriv_pred_pos_idx) / len(true_positive_unpriv_idx)    

    return TPR_unpriv

Define below functions using above

In [77]:
def false_positive_rate_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    FPR_unpriv = false_positive_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    FPR_priv = false_positive_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    FPR_diff = FPR_unpriv - FPR_priv

    return FPR_diff

In [ ]:
def false_negative_rate_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    FNR_unpriv = false_negative_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    FNR_priv = false_negative_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    FNR_diff = FNR_unpriv - FNR_priv  

    return FNR_diff

In [ ]:
def true_positive_rate_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    TPR_diff = - false_negative_rate_difference(y_true, y_pred, prot_attr, priv_group, pos_label)

    return TPR_diff


In [ ]:
def true_negative_rate_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    TNR_diff = - false_positive_rate_difference(y_true, y_pred, prot_attr, priv_group, pos_label)

    return TNR_diff

In [78]:
FPR_diff = false_positive_rate_difference(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test, 
                               priv_group=sens_privileged_groups[0][sens_variable_name], 
                               pos_label=response_favorable_label)
FPR_diff

-0.3458646616541353

In [81]:
FNR_diff = false_negative_rate_difference(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test, 
                               priv_group=sens_privileged_groups[0][sens_variable_name], 
                               pos_label=response_favorable_label)
FNR_diff

0.320855614973262

In [76]:
separation = (np.abs(FPR_diff) + np.abs(FNR_diff)) / 2
separation

np.float64(0.33336013831369865)

#### False/True positive/negative rate ratio

`false_positive_rate_ratio` $= \dfrac{FPR_{A = \text{unpriv}}}{FPR_{A = \text{priv}}}$

`false_negative_rate_ratio` $= \dfrac{FNR_{A = \text{unpriv}}}{FNR_{A = \text{priv}}}$

`true_positive_rate_ratio` $= \dfrac{TPR_{A = \text{unpriv}}}{TPR_{A = \text{priv}}} = \dfrac{1 - FNR_{A = \text{unpriv}}}{1 - FNR_{A = \text{priv}}}$ 

`true_negative_rate_ratio` $= \dfrac{TNR_{A = \text{unpriv}}}{TNR_{A = \text{priv}}} = \dfrac{1 - FPR_{A = \text{unpriv}}}{1 - FPR_{A = \text{priv}}}$

In [ ]:
def false_positive_rate_ratio(y_true, y_pred, prot_attr, priv_group, pos_label):

    FPR_unpriv = false_positive_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    FPR_priv = false_positive_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    FPR_ratio = FPR_unpriv / FPR_priv

    return FPR_ratio


In [ ]:
def false_negative_rate_ratio(y_true, y_pred, prot_attr, priv_group, pos_label):

    FNR_unpriv = false_negative_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    FNR_priv = false_negative_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    FNR_ratio = FNR_unpriv / FNR_priv

    return FNR_ratio

In [ ]:
def true_positive_rate_ratio(y_true, y_pred, prot_attr, priv_group, pos_label):

    TPR_unpriv = true_positive_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    TPR_priv = true_positive_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    TPR_ratio = TPR_unpriv / TPR_priv

    return TPR_ratio


In [ ]:
def true_negative_rate_ratio(y_true, y_pred, prot_attr, priv_group, pos_label):

    TNR_unpriv = true_negative_rate_unprivileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    TNR_priv = true_negative_rate_privileged(y_true, y_pred, prot_attr, priv_group, pos_label)
    TNR_ratio = TNR_unpriv / TNR_priv

    return TNR_ratio
